# bsc_01 — M0 Headroom  ⭐ **GATE 1 (cổng chặn chính)**

**Đây là thứ quan trọng nhất của Stage 1.** Không có trong plan doc. Nó hỏi đúng câu
ROI cascade đã quên: **phần thưởng lớn cỡ nào — TRƯỚC khi xây bất cứ thứ gì.**

Chạy trên **OAI-ZIB test (103 ca)** — nguồn duy nhất có GT xương. ~1h CPU, **không GPU**.

Đo, với mỗi ca × mỗi lớp sụn:
1. Trường độ dày GT (từ tia dọc pháp tuyến xương).
2. Phân rã **error mass** của ResEnc (B0) theo bin độ dày — hai chiều.
3. Counterfactual: nếu triệt tiêu lỗi ở bin {absent, ≤0.5, ≤1.0}, ASSD cải thiện bao nhiêu.
4. M0c: bậc thang z của GT.

**GATE 1:**
- ✅ PROCEED: `ΔASSD_prize ≥ 0.08mm` **và** thin+absent ≥ 35% error mass
- ⚠️ RESCOPE: `ΔASSD_prize ∈ [0.04, 0.08)` → viết lại mục tiêu quanh presence/thickness
- ❌ STOP: `ΔASSD_prize < 0.04mm` **hoặc** bậc thang z chiếm ưu thế

**Xác suất thật thà: 50/50.** Sụn đùi phần lớn dày 2–3mm. Nhưng error mass (không phải
diện tích) mới đáng kể — vùng mỏng khó bất tương xứng.


### Cell config (giống bsc_00)

In [ ]:
# ============================================================
# CELL CONFIG CHUAN - tai dung o MOI notebook bsc_*
# Drive-first: MOI artifact nam duoi BSC_ROOT. KHONG ghi vao /content/.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/<user>/nnUnet-OAI"   # <-- doi thanh repo cua ban
REPO_DIR = "/content/repo"

import os, sys
if not os.path.isdir(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
sys.path.insert(0, REPO_DIR)

# Thu can cho Colab (may local da co scipy/skimage/numpy)
!pip install -q nibabel SimpleITK 2>/dev/null

BSC_ROOT = "/content/drive/MyDrive/bsc"          # goc artifact - TAT CA nam duoi day
os.makedirs(BSC_ROOT, exist_ok=True)
for sub in ["splits","baselines","geom","raydb","atlas","runs"]:
    os.makedirs(f"{BSC_ROOT}/{sub}", exist_ok=True)

# Duong du lieu cu (READ-ONLY - khong bao gio ghi de)
RAW = "/content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA"   # <-- kiem lai duong nay
print("BSC_ROOT =", BSC_ROOT)
print("Cach ly: doc RAW read-only, ghi MOI THU duoi BSC_ROOT, dataset/folder moi.")

## Chạy M0 trên 103 ca test

Cần **prediction B0 trên test set** (không phải CV). Nếu chưa có, chạy inference d020
fold_0 trên `imagesTs` (103 ca) rồi lưu vào `BSC_ROOT/baselines/B0_test_pred/`.
Đây là lần **duy nhất** dùng test set trước Gate 4 — chỉ để đo headroom, không tune gì.

In [ ]:
import glob, numpy as np, json
from bsc import io_utils, headroom, core
from bsc.core import RayConfig

RAW_TS_IMG = f"{RAW}/imagesTs"          # 103 anh test
RAW_TS_LAB = f"{RAW}/labelsTs"          # 103 GT test (co xuong+sun)
B0_PRED    = f"{BSC_ROOT}/baselines/B0_test_pred"   # pred B0 tren test (chay inference neu chua co)

CART = {"femoral_cart":2, "med_tib_cart":4}   # sun dui + chay trong (§3.1)
BONE = {"femoral_cart":1, "med_tib_cart":3}   # xuong tuong ung lam neo he toa do

cfg = RayConfig()
cases = io_utils.list_cases(RAW_TS_IMG)
print(f"{len(cases)} ca test")

results = {c: {"prize":[], "mass":[], "stair":[]} for c in CART}
for cid in cases:
    gt, sp = io_utils.load_nii(f"{RAW_TS_LAB}/{cid}.nii.gz")
    pr, _  = io_utils.load_nii(f"{B0_PRED}/{cid}.nii.gz")
    for c in CART:
        bone = (gt == BONE[c])
        gt_c, pr_c = (gt == CART[c]), (pr == CART[c])
        if not gt_c.any() or not bone.any(): continue
        results[c]["prize"].append(headroom.prize_counterfactual(gt_c, pr_c, bone, sp, cfg))
        results[c]["mass"].append(headroom.error_mass_by_thickness(gt_c, pr_c, bone, sp, cfg))
        results[c]["stair"].append(headroom.gt_staircase_z_vs_inplane(gt_c, sp))
print("Xong M0.")

## GATE 1 — quyết định

In [ ]:
for c in CART:
    g = headroom.evaluate_gate1(results[c]["prize"], results[c]["mass"])
    print(f"\n=== {c} ===")
    print(f"  ΔASSD_prize = {g['d_assd_prize_mean_mm']:.4f} mm  CI {g['d_assd_prize_ci']}")
    print(f"  thin+absent error mass = {g['thin_absent_error_mass_frac_mean']:.1%}")
    print(f"  QUYET DINH: {g['decision']}")

    # Bang error mass theo bin
    import numpy as np
    print(f"  {'bin':<10}{'N':>8}{'mean_d':>9}{'mass%':>8}")
    for nm in core.THICKNESS_NAMES:
        fr = np.mean([m['per_bin'][nm]['frac_error_mass'] for m in results[c]['mass']])
        nn = np.mean([m['per_bin'][nm]['n'] for m in results[c]['mass']])
        md_ = np.mean([m['per_bin'][nm]['mean_dist_mm'] for m in results[c]['mass']])
        print(f"  {nm:<10}{nn:>8.0f}{md_:>9.3f}{fr:>7.1%}")

    stair = np.nanmean([s['z_over_inplane_std'] for s in results[c]['stair']])
    print(f"  M0c bac thang z/in-plane = {stair:.3f}  (>>1 => nhieu z chiem uu the)")

## Ghi kết quả M0 vào Drive (§7)

Dù quyết định là gì, **lưu lại** — kết quả âm tính M0 vẫn publishable (cùng ROI cascade).

In [ ]:
import numpy as np
now = None   # notebook: co the dat chuoi thoi gian thu cong neu muon
out = {c: {"gate1": headroom.evaluate_gate1(results[c]["prize"], results[c]["mass"])}
       for c in CART}
# ep numpy -> float cho json
def clean(o):
    if isinstance(o, dict): return {k:clean(v) for k,v in o.items()}
    if isinstance(o, (list,tuple)): return [clean(x) for x in o]
    if isinstance(o, (np.floating,np.integer)): return float(o)
    return o
path = f"{BSC_ROOT}/runs/M0_headroom_B0_ zibTs.json".replace(" ","")
json.dump(clean(out), open(path,"w"), indent=2)
print("Da ghi", path)
print("\nNEU PROCEED -> Phase 2 (geometry QC: M3/M2/M4 tren xuong that).")
print("NEU RESCOPE  -> viet lai muc tieu §3.7 quanh presence F1 + thickness MAE.")
print("NEU STOP     -> cong bo ket qua am tinh. Khong dot them chu ky.")